# Generate Political Labels for `ukr_rus_twitter`

This notebook derives weak political labels from retweet targets and profile-description keywords, then writes a labeled graph artifact for classification experiments.

The default output is a copy of the existing graph with `y` and `label_names` populated.

In [ ]:
import csv
import glob
import os
from collections import defaultdict

import numpy as np
import pandas as pd
import torch

GRAPH_PATH = "/scratch1/eibl/data/ukr_rus_twitter/graphs/retweet_graph_150files_minilm_hf03.pt"
OUT_PATH = "/scratch1/eibl/data/ukr_rus_twitter/graphs/retweet_graph_150files_minilm_hf03_political_labels.pt"
CSV_GLOB = "/project2/ll_774_951/uk_ru/twitter/data/*/*.csv"
MAX_FILES = 150

PRO_UKRAINE_TARGETS = {
    "ukraine", "zelenskyyua", "kyivindependent", "mfa_ukraine", "defenceu", "ukrainenow", "uamemesforces"
}
PRO_RUSSIA_TARGETS = {
    "kremlinrussia_e", "mfa_russia", "rt_com", "sputnikint", "medvedevrussiae", "mod_russia", "readovkanews"
}

PRO_UKRAINE_KEYWORDS = [
    "standwithukraine", "slavaukraine", "slavaukraini", "supportukraine", "russiaisaterroriststate", "stoprussia"
]
PRO_RUSSIA_KEYWORDS = [
    "standwithrussia", "istandwithrussia", "istandwithputin", "russiaisnotalone", "zov", "forrussia"
]

MIN_SCORE = 2
SCORE_MARGIN = 2


In [ ]:
def normalize_handle(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip().str.lower()
    return s.mask(s.isin(["", "nan", "none", "<na>"]))


def load_interleaved_csv(filepath: str) -> pd.DataFrame:
    main_rows, sub_rows = [], []
    with open(filepath, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
            sub_header_raw = next(reader)
        except StopIteration:
            return pd.DataFrame()

    with open(filepath, "r", encoding="utf-8", errors="replace", newline="") as f:
        reader = csv.reader(f)
        next(reader)
        if sub_header_raw is not None:
            next(reader)
        pending_main = None
        for row in reader:
            if not row:
                continue
            if len(row) == 66:
                if pending_main is not None:
                    main_rows.append(pending_main)
                    sub_rows.append([""] * 11)
                pending_main = row
            elif len(row) == 11:
                if pending_main is not None:
                    main_rows.append(pending_main)
                    sub_rows.append(row)
                    pending_main = None
            else:
                continue
        if pending_main is not None:
            main_rows.append(pending_main)
            sub_rows.append([""] * 11)

    sub_cols = [
        "sub_extra", "state", "country", "rt_state", "rt_country", "qtd_state", "qtd_country",
        "norm_country", "norm_rt_country", "norm_qtd_country", "acc_age",
    ]
    df_main = pd.DataFrame(main_rows, columns=header)
    df_sub = pd.DataFrame(sub_rows, columns=sub_cols).drop(columns=["sub_extra"], errors="ignore")
    return pd.concat([df_main.reset_index(drop=True), df_sub.reset_index(drop=True)], axis=1)


def load_rows(csv_glob: str, max_files: int) -> pd.DataFrame:
    files = sorted(glob.glob(csv_glob))[:max_files]
    usecols = ["screen_name", "rt_screen", "description"]
    chunks = []
    for i, path in enumerate(files, start=1):
        print(f"[{i}/{len(files)}] {os.path.basename(path)}")
        df = load_interleaved_csv(path)
        if df.empty:
            continue
        cols = [c for c in usecols if c in df.columns]
        if not cols:
            continue
        chunks.append(df[cols].copy())
    if not chunks:
        raise RuntimeError("No rows loaded")
    out = pd.concat(chunks, ignore_index=True)
    out["screen_name"] = normalize_handle(out["screen_name"])
    if "rt_screen" in out.columns:
        out["rt_screen"] = normalize_handle(out["rt_screen"])
    out["description"] = out.get("description", "").fillna("").astype(str).str.lower()
    out = out[out["screen_name"].notna()].copy()
    return out


In [ ]:
graph = torch.load(GRAPH_PATH, map_location="cpu")
handles = graph["handles"]
h2i = graph["h2i"]
print(f"graph nodes: {len(handles):,}")

rows = load_rows(CSV_GLOB, MAX_FILES)
print(f"loaded rows: {len(rows):,}")

rt = rows[["screen_name", "rt_screen"]].dropna().copy()
rt = rt[rt["screen_name"] != rt["rt_screen"]].copy()
uk_rt = rt[rt["rt_screen"].isin(PRO_UKRAINE_TARGETS)].groupby("screen_name").size().rename("uk_rt")
ru_rt = rt[rt["rt_screen"].isin(PRO_RUSSIA_TARGETS)].groupby("screen_name").size().rename("ru_rt")

desc = rows[["screen_name", "description"]].drop_duplicates(["screen_name", "description"]).copy()
uk_kw = desc["description"].apply(lambda s: sum(k in s for k in PRO_UKRAINE_KEYWORDS))
ru_kw = desc["description"].apply(lambda s: sum(k in s for k in PRO_RUSSIA_KEYWORDS))
desc_scores = pd.DataFrame({"screen_name": desc["screen_name"], "uk_kw": uk_kw, "ru_kw": ru_kw})
desc_scores = desc_scores.groupby("screen_name", as_index=True).sum()

scores = pd.DataFrame(index=sorted(set(handles)))
scores = scores.join(uk_rt, how="left").join(ru_rt, how="left").join(desc_scores, how="left").fillna(0)
scores["uk_score"] = scores["uk_rt"] + 3 * scores["uk_kw"]
scores["ru_score"] = scores["ru_rt"] + 3 * scores["ru_kw"]
scores["margin"] = scores["uk_score"] - scores["ru_score"]

label_names = ["pro_russia", "pro_ukraine"]
y = torch.full((len(handles),), -1, dtype=torch.long)

pro_ukraine = scores[(scores["uk_score"] >= MIN_SCORE) & (scores["margin"] >= SCORE_MARGIN)].index
pro_russia = scores[(scores["ru_score"] >= MIN_SCORE) & (scores["margin"] <= -SCORE_MARGIN)].index

for handle in pro_russia:
    idx = h2i.get(handle)
    if idx is not None:
        y[idx] = 0
for handle in pro_ukraine:
    idx = h2i.get(handle)
    if idx is not None:
        y[idx] = 1

print("label counts:")
print("  pro_russia:", int((y == 0).sum().item()))
print("  pro_ukraine:", int((y == 1).sum().item()))
print("  unlabeled:", int((y < 0).sum().item()))


In [ ]:
graph["y"] = y
graph["label_names"] = label_names
if "data" in graph:
    graph["data"].y = y
    graph["data"].label_names = label_names

torch.save(graph, OUT_PATH)
print("saved:", OUT_PATH)
